# Week 36, Day 6: Forge: Week 36 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(np.degrees(1 / np.sqrt(400)), 2), "deg;", round(np.degrees(1 / np.sqrt(1600)), 2), "deg with four times the window")
print("2.", round(300 / (1 + 0.01 * 300), 1), "(the limit is 1/ε = 100)")
print("3.", 4 / (4 + 1))

### Exercise: Part A: decoder and bound

Write bound_deg(N): the Cramér-Rao standard deviation in degrees, 1 / sqrt(T Σ f'(θ)² / f(θ)), for a population built by pop(N), at θ = 0.3 (use a central difference for f'). The decoding loop is given.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(36)
grid = np.linspace(0, 2 * np.pi, 360, endpoint=False)
T, sizes = 0.1, (10, 30, 100, 300, 1000)

def pop(N, kappa=3.0, rmax=30.0, b=1.0):
    pref = np.linspace(0, 2 * np.pi, N, endpoint=False)
    return lambda th: b + rmax * np.exp(kappa * (np.cos(np.asarray(th)[..., None] - pref) - 1))

def bound_deg(N, th0=0.3, e=1e-5):
    f = pop(N)
    ...

def decode_rmse(N, mode, trials=1500):
    f = pop(N); F = f(grid) * T; lF = np.log(F); S = F.sum(1); errs = []
    for _ in range(trials):
        th = r.uniform(0, 2 * np.pi); g, th_in = 1.0, th
        if mode == "gain":
            g = np.exp(0.4 * r.normal() - 0.08)
        if mode == "jitter":
            th_in = th + np.radians(3) * r.normal()
        n = r.poisson(g * f(th_in) * T)
        errs.append(np.angle(np.exp(1j * (grid[np.argmax(lF @ n - S)] - th))))
    return np.degrees(np.sqrt(np.mean(np.square(errs))))

bound = {N: bound_deg(N) for N in sizes}
rmse = {"independent": {N: decode_rmse(N, "independent") for N in sizes}}
for N in sizes:
    print(f"N = {N:4d}: bound {bound[N]:5.2f} deg, decoder {rmse['independent'][N]:5.2f} deg")

In [ ]:
import numpy as np
def test_bound_fn():
    "bound_deg(N) falls as one over the square root of N"
    assert abs(bound_deg(400) / bound_deg(100) - 0.5) < 0.02

def test_efficient():
    "for 30 or more cells, the decoder's error is within 15% of the bound"
    for N in (30, 100, 300, 1000):
        assert rmse["independent"][N] < 1.15 * bound[N]

test_bound_fn()
test_efficient()
print("All checks passed")

### Exercise: Part B: gain against input noise

Fill in rmse['gain'] and rmse['jitter'] for every population size, using decode_rmse with the matching mode, and plot all three conditions against the bound on log-log axes.

In [ ]:
for mode in ("gain", "jitter"):
    rmse[mode] = ...
for mode, v in rmse.items():
    plt.loglog(sizes, [v[N] for N in sizes], "o-", label=mode)
    print(mode, {N: round(float(v[N]), 2) for N in sizes})
plt.loglog(sizes, [bound[N] for N in sizes], "k--", label="bound, independent")
plt.axhline(3, color="gray", ls=":"); plt.xlabel("cells"); plt.ylabel("RMS error (deg)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_gain():
    "with 1,000 cells, shared gain noise raises the error by less than 30%"
    assert rmse["gain"][1000] < 1.3 * rmse["independent"][1000]

def test_jitter():
    "with 1,000 cells, input jitter at least doubles the error and leaves it above 2.5 degrees"
    assert rmse["jitter"][1000] > 2 * rmse["independent"][1000] and rmse["jitter"][1000] > 2.5

test_gain()
test_jitter()
print("All checks passed")